In [13]:
import time
import re
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

In [ ]:
def fetch_banapresso():
    url = "https://www.banapresso.com/"

    chrome_options = Options()
    prefs = {
        "profile.default_content_setting_values.geolocation": 2
    }
    chrome_options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=chrome_options) 
    driver.maximize_window()

    driver.get(url)
    time.sleep(1)

    action = ActionChains(driver)



    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#wrap > header > div > ul > li:nth-child(2) > a > span"
    )
    
    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#wrap > header > div > ul > li:nth-child(2) > ul > li:nth-child(1) > a > span"
    )

    action.move_to_element(first_tag) \
          .move_to_element(second_tag) \
          .click() \
          .perform()
    
    time.sleep(2)

    driver.find_element(By.XPATH, "//*[@id=\"root\"]/div[2]/div/div[2]/button").click()
    
    # 여기가지 매장 찾기 열기 및 위치 팝업까지 해결
    # -------------------------------------------------------------------------------------------------

    scroll_area = driver.find_element(By.XPATH,"//*[@id=\"store\"]/div/div[1]/div[3]")
    before = 0

    while True:
        driver.execute_script(
            "arguments[0].scrollTop = arguments[0].scrollHeight",
            scroll_area
        )

        time.sleep(1)

        after = driver.execute_script(
            "return arguments[0].scrollHeight",
            scroll_area
        )
        if before == after:
            break

        before = after
    
    # print("스크롤 끝") # 스크롤 확인



    # 데이터 가져오기
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "store_shop_list") 
        )
    )
    
    time.sleep(1)


    req = driver.page_source
    
    soup = BeautifulSoup(req,"html.parser")

    stores = soup.find('ul').find_all('li')
    

    store_list = []
    addr_list = []
    open_close_list = []
    

    for store in stores:
        name = store.select_one("p.name")
        addr = store.select_one("p.address")
        octime = store.select_one("div.store-time-wrap div.time.close p")

        store_name = name.text.strip() if name else ""
        store_addr = addr.text.strip() if addr else ""
        store_open_close = "".join(octime.stripped_strings) if octime else "안내없음"

        store_list.append(store_name)
        addr_list.append(store_addr) 
        open_close_list.append(store_open_close)

    
    df = pd.DataFrame({
        '가게명': store_list,
        '주소': addr_list,
        '영업시간': open_close_list
    })

    driver.quit()

    return df


In [81]:
banapresso_df = fetch_banapresso()

# CSV 저장
banapresso_df.to_csv(
    "banapresso_location.csv",
    index=False,
    encoding='utf-8-sig'
)

print("데이터가 banapresso_location.csv 파일로 저장되었습니다.")
print(banapresso_df.head())

데이터가 banapresso_location.csv 파일로 저장되었습니다.
         가게명                             주소         영업시간
0       가락몰점  서울특별시 송파구 양재대로 932, 업무동 1층 로비  07:30~23:30
1  가산디지털단지역점               서울시 금천구 가산동 60-3  07:00~19:00
2     가산안양천점  서울 금천구 가산 디지털2로 127-143, 101호  07:00~20:00
3    가산어반워크점   서울시 금천구 가산디지털2로 135, 1동 142호  07:00~17:30
4  가산우림라이온스점       서울 금천구 가산디지털1로 168 b131호  07:00~17:00
